In [48]:
import pandas as pd 
import numpy as np 

import joblib

import glob
import os

from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import confusion_matrix, classification_report


import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense

In [49]:
import pandas as pd
import glob
from sklearn.preprocessing import LabelEncoder

# Find all CSV files in current folder
csv_files = glob.glob("*.csv")

print("Found CSV files:")
for file in csv_files:
    print("-", file)

# Load and combine all CSV files
dfs = []

for file in csv_files:
    try:
        temp_df = pd.read_csv(file)
        dfs.append(temp_df)
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Combine into one DataFrame
df = pd.concat(dfs, ignore_index=True)

# Remove any completely empty rows
df = df.dropna()

# Ensure numeric columns are numeric
df["sound"] = pd.to_numeric(df["sound"], errors="coerce")
df["light"] = pd.to_numeric(df["light"], errors="coerce")

# Remove bad rows that couldn't be converted
df = df.dropna()

# Convert string labels to integers
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label"])

print("\nLabel Mapping:")
for i, label in enumerate(encoder.classes_):
    print(f"{label} -> {i}")

print("\nTotal rows:", len(df))
print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
print(df.head())



Found CSV files:
- dark_20260614_112619.csv
- dark_20260614_112827.csv
- focus_20260614_111410.csv
- focus_20260614_121237.csv
- noisy_20260614_113446.csv
- noisy_20260614_121650.csv

Label Mapping:
dark -> 0
focus -> 1
noisy -> 2

Total rows: 613

Data Types:
sound    float64
light    float64
label      int32
dtype: object

First 5 Rows:
   sound  light  label
0  10.19    3.0      0
1  14.15    3.0      0
2  22.99    3.0      0
3  17.93    3.0      0
4  21.81    3.0      0


In [50]:
X = df[["sound", "light"]].astype("float32")
y = df["label"].astype("int32")  
X_train, X_test, y_train, y_test = train_test_split( 
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y 
)

In [51]:
scaler = StandardScaler() 
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test)

In [52]:
joblib.dump(scaler, "scaler.pkl")

print("Scaler saved")

Scaler saved


In [53]:
model = Sequential([ 
    Dense(16, activation="relu", input_shape=(2,)), 
    Dense(16, activation="relu"), 
    Dense(3, activation="softmax") 
    ]) 

model.compile( 
    optimizer="adam", 
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"] 
)

model.summary()

c:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 16)             │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 371 (1.45 KB)

 Trainable params: 371 (1.45 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
history = model.fit( 
    X_train, 
    y_train, 
    epochs=50, 
    batch_size=16, 
    validation_split=0.2 
)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.4362 - loss: 1.0170 - val_accuracy: 0.7857 - val_loss: 0.9619
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7908 - loss: 0.9014 - val_accuracy: 0.8265 - val_loss: 0.8543
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8291 - loss: 0.8129 - val_accuracy: 0.8980 - val_loss: 0.7599
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8673 - loss: 0.7285 - val_accuracy: 0.8980 - val_loss: 0.6766
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8878 - loss: 0.6494 - val_accuracy: 0.8980 - val_loss: 0.5996
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9082 - loss: 0.5789 - val_accuracy: 0.8980 - val_loss: 0.5307
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9133 - loss: 0.5164 - val_accuracy: 0.8980 - val_loss: 0.4752
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9184 - loss: 0.4623 - val_accuracy: 0.8980 - val_los

In [55]:
loss, accuracy = model.evaluate(X_test, y_test) 

print("Test Accuracy:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9675 - loss: 0.0974
Test Accuracy: 0.9674796462059021


In [56]:


predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)

print(confusion_matrix(y_test, predicted_labels))
print(classification_report(y_test, predicted_labels))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
[[26  0  2]
 [ 0 30  1]
 [ 1  0 63]]
              precision    recall  f1-score   support

           0       0.96      0.93      0.95        28
           1       1.00      0.97      0.98        31
           2       0.95      0.98      0.97        64

    accuracy                           0.97       123
   macro avg       0.97      0.96      0.97       123
weighted avg       0.97      0.97      0.97       123



In [57]:
model.save("room_classifier.h5")

In [58]:
converter = tf.lite.TFLiteConverter.from_keras_model(model) 
tflite_model = converter.convert() 

with open("room_classifier.tflite", "wb") as f: 
    f.write(tflite_model) 
    
print("TFLite model saved")

INFO:tensorflow:Assets written to: C:\Users\tebur\AppData\Local\Temp\tmpc53817o5\assets


INFO:tensorflow:Assets written to: C:\Users\tebur\AppData\Local\Temp\tmpc53817o5\assets


Saved artifact at 'C:\Users\tebur\AppData\Local\Temp\tmpc53817o5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor_20')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  3079581245584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3079581244816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3079581248080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3079581250768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3079666425232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3079666421008: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved


In [59]:
with open("room_classifier.tflite", "rb") as f:
    data = f.read()

with open("room_classifier.h", "w") as f:

    f.write("const unsigned char room_classifier_tflite[] = {\n")

    for i, byte in enumerate(data):

        if i % 12 == 0:
            f.write("    ")

        f.write(f"0x{byte:02x},")

        if i % 12 == 11:
            f.write("\n")
        else:
            f.write(" ")

    f.write("\n};\n")

    f.write(
        f"\nconst unsigned int room_classifier_tflite_len = {len(data)};\n"
    )

print("Header generated")

Header generated
